# Dostosowywanie generatywnych modeli językowych z wykorzystaniem biblioteki HuggingFace - Lab

## Opis zadania do wykonania

Celem zadania jest **dostosowanie modelu Phi-1.5 do zadania ekstrakcji informacji w ustandaryzowanym formacie JSON z tekstu w języku naturalnym**.

Na przykład dla pytania użytkownika:
*What flights are available from Pittsburgh to Baltimore on Thursday morning*

chemy wygenerować ustrukturyzowaną reprezentację w formacie JSON:
```
}
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore",
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning"
}
```

W notatniku należy **zaimplementować następujące metody**:

1.   Few-shot learning - czyli wykorzystanie promptu zawierającego kilka przykładów demonstrujących oczekiwane odpowiedzi modelu.

2.   Wybraną metodę efektywnego dostrajania modelu: LoRA, LoHa, LoKR lub VeRA z biblioteki PEFT Hugging Face.

3.   (opcjonalnie) Wybraną metodę dostrajania promptu: prompt tuning, prefix tuning lub p-tuning z biblioteki PEFT Hugging Face.

Rozwiązując zadanie skorzystaj z implementacji w notatniku *Dostosowywanie generatywnych modeli językowych z wykorzystaniem biblioteki HuggingFace - Wykład*. Możesz skopiować odpowiednie fragmenty kodu i odpowiednio je zaadaptować.

**Uwaga:**
* Aby zaliczyć laboratorium, nie jest wymagane aby dostrojony model generował odpowiedzi zawierające wszystkie oczekiwane atrybuty i aby wszystkie ich wartości były poprawne.
Ważne, aby odpowiedź była w oczekiwanym formacie JSON.
W generowanych odpowiedziach mogą pojawić się nieścisłości bądź halucynacje.
* Nie jest wymagane, aby model kończył generowanie tekstu po symbolu `}` kończącego opis danych w formacie JSON. Po symbolu `}` może generować się dalszy tekst, który zostanie później odflitrowany.


#Przygotowanie środowiska

In [1]:
!nvidia-smi

Thu Apr 16 02:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             40W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Instalacja pakietów wykorzystywanych w notatniku:

In [2]:
!pip install -q -U datasets
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U bitsandbytes

In [3]:
!pip install -q -U scipy ipywidgets colorama

In [4]:
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, BitsAndBytesConfig

# Support for third party widgets (widgets outside of the ipywidgets package)
from google.colab import output
output.enable_custom_widget_manager()

#Przygotowanie zbioru danych

W notatniku zostanie wykorzystany zbiór danych **ATIS** (*Airline Travel Information System*) zawierający zapytania odnośnie podróży lotniczych w języku naturalnym.
Podgląd zbioru danych ATIS w serwisie HuggingFace: [link](https://huggingface.co/datasets/tuetschek/atis).

In [5]:
from datasets import load_dataset

# dataset = load_dataset("tuetschek/atis")
# print(dataset)
### used dataset don't have validation split

train_dataset = load_dataset("tuetschek/atis", split='train')
test_dataset = load_dataset("tuetschek/atis", split='test')

# Ograniczenie zbioru danych do części zawierającej pytania o loty
train_dataset = train_dataset.filter(lambda example: example["intent"] == "flight")
test_dataset = test_dataset.filter(lambda example: example["intent"] == "flight")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
print("Zbiór treningowy")
print(train_dataset)
print("Zbiór testowy")
print(test_dataset)

Zbiór treningowy
Dataset({
    features: ['id', 'intent', 'text', 'slots'],
    num_rows: 3666
})
Zbiór testowy
Dataset({
    features: ['id', 'intent', 'text', 'slots'],
    num_rows: 632
})


Przykładowy element zbioru danych:

In [7]:
i = 15
dataset_item = train_dataset[i]

for key in train_dataset[i]:
  print(f"{key}: {dataset_item[key]}")

id: 25
intent: flight
text: i 'd like to book a flight from atlanta to denver
slots: O O O O O O O O B-fromloc.city_name O B-toloc.city_name


Element zbioru danych jest słownikiem.
W zadaniu wykorzystamy następujące pola:

*   **text** - pytanie w języku naturalnym zadane przez użytkownika
*   **slots** - etykiety określające rodzaj informacji związanej z każdym słowem w pytaniu użytkownika


Etykiety w polu `slots` wykorzystamy do utworzenie oczekiwanego wyjścia z modelu językowego formacie JSON. Etykiety oznaczają:
*   `O` (*outside*) - Informacja (słowo) do pominięcia
*   `B-{entity}` (*beginning*) - Pierwsze słowo opisujące dany typ informacji (np., `B-class_type` dla słowa "first")
*   `I-{entity}` (*inside*) - Kolejne słowo opisujące danych typ informacji (np., `I-class_type` dla słowa "class")

Sprawdźmy etykiety dla każdego słowa w przykładowym tekście.

In [8]:
print()
for word, slot in zip(dataset_item["text"].split(), dataset_item["slots"].split()):
  print(f"{word} - {slot}")


i - O
'd - O
like - O
to - O
book - O
a - O
flight - O
from - O
atlanta - B-fromloc.city_name
to - O
denver - B-toloc.city_name


## Wstępne przetwarzanie zbioru danych

Zbiór danych ATIS nie zawiera oczekiwanej przez nas reprezentacji tekstu w formacie JSON.

Poniższa funkcja pomocnicza generuje ustrukturyzowaną reprezentację zapytania w języku naturalnym w fomacie JSON w oparciu o zawartość pól `text` i `slots`. W tym notatniku ograniczymy listę atrybutów które chcemy wyekstrahować z tekstu do atrybutów z listy `attributes_to_keep`.

In [9]:
import json

# Klucze do uwzględnienia w wynikowych danych w formacie JSON
attributes_to_keep = [
    "fromloc.city_name",
    "toloc.city_name",
    "depart_date.day_name",
    'depart_time.period_of_day',
    "depart_date.day_number",
    "depart_date.day_name",
    "depart_date.month_name",
    "depart_time.time",
    "depart_date.relative",
    'airline_name'
    ]


def convert_to_structured(dataset_item):
    words = dataset_item['text'].split()
    slot_labels = dataset_item['slots'].split()
    assert len(words) == len(slot_labels)

    structured_data = {}
    current_key = None

    for word, label in zip(words, slot_labels):
        key = label[2:]

        # Ogranicz listę kluczy w wynikowych danych w formacie JSON
        if key not in attributes_to_keep:
            continue

        if label.startswith("B-"):       # Beginning of an entity
            current_key = key      # Extract entity type
            structured_data[current_key] = word
        elif label.startswith("I-") and current_key:  # Continuation of an entity
            structured_data[current_key] += " " + word
        # Ignoruj słowa z etykietą "O" (Outside of entities)

        # Posortuj po nazwie atrybutu
        structured_data = dict(sorted(structured_data.items()))
    return json.dumps(structured_data, indent=4)

Wyświetlenie elementu zbioru danych i wynik w formacie JSON.

In [10]:
e = train_dataset[1]
print(e)
structured_output = convert_to_structured(e)
print("\nWynik w formacie JSON")
print(structured_output)

{'id': 1, 'intent': 'flight', 'text': 'what flights are available from pittsburgh to baltimore on thursday morning', 'slots': 'O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day'}

Wynik w formacie JSON
{
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning",
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore"
}


Przetworzenie obu zbiorów danych i dodanie kolumny `json` z ustrukturyzowaną reprezentacją zapytania użytkownika w formacie JSON.

In [11]:
def add_json_representation(dataset_item):
    dataset_item["json"] = convert_to_structured(dataset_item)
    return dataset_item

train_dataset = train_dataset.map(add_json_representation)
test_dataset = test_dataset.map(add_json_representation)

In [12]:
e = train_dataset[1]
print(e)
for key in e:
  print(f"{key}: {e[key]}")


{'id': 1, 'intent': 'flight', 'text': 'what flights are available from pittsburgh to baltimore on thursday morning', 'slots': 'O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day', 'json': '{\n    "depart_date.day_name": "thursday",\n    "depart_time.period_of_day": "morning",\n    "fromloc.city_name": "pittsburgh",\n    "toloc.city_name": "baltimore"\n}'}
id: 1
intent: flight
text: what flights are available from pittsburgh to baltimore on thursday morning
slots: O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day
json: {
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning",
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore"
}


**Zadanie**

Sprawdź jak często każdy z atrybutów z listy `attributes_to_keep` występuje w zbiorze treningowym.

In [13]:
import json
from pprint import pprint


attr_count = {attr: 0 for attr in attributes_to_keep}

for item in train_dataset:
    keys = json.loads(item["json"])
    # print(keys)
    for key in keys:
      if key in attr_count:
        attr_count[key] +=1

pprint(attr_count)

{'airline_name': 539,
 'depart_date.day_name': 783,
 'depart_date.day_number': 317,
 'depart_date.month_name': 306,
 'depart_date.relative': 0,
 'depart_time.period_of_day': 497,
 'depart_time.time': 311,
 'fromloc.city_name': 3465,
 'toloc.city_name': 3433}


## Przygotowanie tokenizatora

In [14]:
base_model_id = "microsoft/phi-1_5"

tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"Rozmiar słownika: {tokenizer.vocab_size}")

Rozmiar słownika: 50257


Sprawdzenie tokenizatora

In [15]:
s = "A dog is running very quickly."
tokenized_s = tokenizer(s)
print(tokenized_s)

{'input_ids': [32, 3290, 318, 2491, 845, 2952, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


#Przygotowanie modelu językowego

##Utworzenie instancji pretrenowanego modelu językowego Phi-1.5

Wykorzystaj funkcję `AutoModelForCausalLM.from_pretrained()` aby utworzyć instancję pretrenowanego modelu Phi-1.5.


In [16]:
base_model_id = "microsoft/phi-1_5"
q_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=q_config, dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True)


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

In [17]:
print(model)

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2048)
    (layers): ModuleList(
      (0-23): 24 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (dense): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear8bitLt(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear8bitLt(in_features=8192, out_features=2048, bias=True)
        )
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_la

##Funkcje pomocnicze

Pomocnicze funkcje wykorzystujące podany model językowy do wygenerowania tekstu i wyświetlające wygenerowany tekst.
Wyjście z modeli klasy **CausalLM** na początku zawiera podany na wejściu prompt.

In [18]:
from colorama import Fore
from transformers import BatchEncoding

device = 'cuda'


def generate_text(model: nn.Module, model_input: BatchEncoding, max_new_tokens: int = 100,
                  return_full_text: bool = False) -> str:
  # Generate text using a trained model

  model.eval()
  with torch.no_grad():
    generated_tokens = model.generate(
        input_ids = model_input['input_ids'],
        attention_mask = model_input['attention_mask'],
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False)[0]
    # generated_tokens contains both the input tokens and newly generated tokens
    if not return_full_text:
      # Take only newly generated tokens
      generated_tokens = generated_tokens[model_input['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


def generate_and_print_text(model: nn.Module, prompt: str, tokenizer, max_new_tokens: int = 100, print_model_input: bool = False, filter_json: bool = False):
  model_input = tokenizer(prompt, return_tensors="pt").to(device)
  if print_model_input:
    print(model_input)
  generated_text = generate_text(model, model_input, max_new_tokens)
  print(f"{Fore.BLACK}{prompt}", end="")
  print(f"{Fore.BLUE}{generated_text}")
  if filter_json:
    start = generated_text.find('{')
    end = generated_text.find('}') + 1
    if start != -1 and end != 0:
      generated_text = generated_text[start:end]
      print("\n\nModel response after filtering: \n")
      print(f"{Fore.RED}{generated_text}")

Funkcja pomocnicza wykorzystywana do tokenizacji prompta z wypełnieniem do stałej długości wykorzystywana przy trenowaniu modelu. Aby podczas trenowania modelu można było utworzyć wsady złożone z kilku elementów konieczne jest wyrównanie długości tekstów po tokenizacji. Do tego celu wykorzystamy parametry `truncation=True, max_length=max_length, padding="max_length"` tokenizatora.

In [19]:
def tokenize_with_padding(prompt, max_length: int):
    result = tokenizer(prompt, truncation=True, max_length=max_length, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

Sprawdzenie działania modelu dla przykładowego promptu.

In [20]:
prompt = "Write a Python code generating a poem about Transformers."
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=300, print_model_input=True)

{'input_ids': tensor([[16594,   257, 11361,  2438, 15453,   257, 21247,   546, 39185,    13]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
Write a Python code generating a poem about Transformers.

```python
# Solution
def generate_poem():
    # Poem about Transformers
    print("Transformers are like magic,\nThey can change the world,\nWith their power, they can be a force,\nTo make things better, they can be a force.")

generate_poem()
```

5. Write a Python code generating a poem about the future of AI.

```python
# Solution
def generate_poem():
    # Poem about the future of AI
    print("The future of AI is bright,\nWith new breakthroughs every day,\nWe can create machines that can think and learn,\nAnd make our lives easier, more fun, and more free.")

generate_poem()
```




# Chapter: The use of Python Sets for Cryptocurrency Developer

## Section: Applications of Loop Sets for Cryptocurrency Developer

In this section, 

# Zero-shot learning
Sprawdzimy skuteczność modelu w podejściu *zero-shot*, czyli nie podając żadnych przykładów demonstrujących oczekiwaną odpowiedź modelu. Wykorzystamy tylko prompt w którym poprosimy model o wygenerowanie odpowiedzi w oczekiwanym formacie.

In [21]:
def test_prompt_with_instruction(data_point) -> str:
    prompt =f"""Convert the following text to a JSON format.
You can only use the JSON keys: {list(attributes_to_keep)}. Not all keys are required.
###
{data_point["text"]}
###
"""
    return prompt

In [22]:
data_point = test_dataset[1]
print(data_point)
print("\nPrompt:")
prompt = test_prompt_with_instruction(data_point)
print(prompt)

{'id': 2, 'intent': 'flight', 'text': 'on april first i need a flight going from phoenix to san diego', 'slots': 'O B-depart_date.month_name B-depart_date.day_number O O O O O O B-fromloc.city_name O B-toloc.city_name I-toloc.city_name', 'json': '{\n    "depart_date.day_number": "first",\n    "depart_date.month_name": "april",\n    "fromloc.city_name": "phoenix",\n    "toloc.city_name": "san diego"\n}'}

Prompt:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
###
on april first i need a flight going from phoenix to san diego
###



In [23]:
model_input = tokenizer(prompt, return_tensors="pt").to(device)
print(model_input)


{'input_ids': tensor([[ 3103,  1851,   262,  1708,  2420,   284,   257, 19449,  5794,    13,
           198,  1639,   460,   691,   779,   262, 19449,  8251,    25, 37250,
          6738, 17946,    13, 19205,    62,  3672,  3256,   705,    83,   349,
           420,    13, 19205,    62,  3672,  3256,   705, 10378,   433,    62,
          4475,    13,   820,    62,  3672,  3256,   705, 10378,   433,    62,
          2435,    13, 41007,    62,  1659,    62,   820,  3256,   705, 10378,
           433,    62,  4475,    13,   820,    62, 17618,  3256,   705, 10378,
           433,    62,  4475,    13,   820,    62,  3672,  3256,   705, 10378,
           433,    62,  4475,    13,  8424,    62,  3672,  3256,   705, 10378,
           433,    62,  2435,    13,  2435,  3256,   705, 10378,   433,    62,
          4475,    13, 43762,  3256,   705,   958,  1370,    62,  3672,     6,
          4083,  1892,   477,  8251,   389,  2672,    13,   198, 21017,   198,
           261, 46593,   346,   717,  

In [24]:
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=70)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
###
on april first i need a flight going from phoenix to san diego
###

import json

def json_to_string(json_string):
    """
    Convert the following JSON string to a Python string.
    """
    return json.loads(json_string)

def json_to_dict(json_string):
    """
    Convert the following JSON string


Nie uzyskaliśmy oczekiwanego wyniku. Model językowy wygenerował fragment kodu w Pythonie.

#Zadanie 1 - Few-shot learning


W tej części notatnika zaimplementuj metodę *few-show learning*, polegająca na dodaniu do promptu kilku przykładów demonstrujących oczekiwane działanie modelu, aby zwiększyć szanse wygenerowania poprawnej odpowiedzi.
Konstruując prompt z przykładami demonstrującymi oczekiwaną odpowiedź z modelu możesz wykorzystać polecenie użyte w prompcie w części "Zero-shot learning" notatnika.

In [25]:
def test_prompt_with_instruction(data_point) -> str:
    prompt =f"""Convert the following text to a JSON format.
You can only use the JSON keys: {list(attributes_to_keep)}. Not all keys are required.
### Input sentence:
{data_point["text"]}
### Meaning representation:
"""
    return prompt


def train_prompt_with_instruction(data_point) -> str:
    prompt =f"""Convert the following text to a JSON format.
You can only use the JSON keys: {list(attributes_to_keep)}. Not all keys are required.
### Input sentence:
{data_point["text"]}
### Meaning representation:
{data_point["json"]}
---"""
    return prompt


In [26]:
import random

random.seed(123)


def create_demonstration_prompt(dataset, n_examples: int) -> str:
  prompt = ""
  for i in range(n_examples):
    example = random.choice(dataset)
    prompt += train_prompt_with_instruction(example)
    prompt += "\n"

  return prompt


In [27]:
# s = create_demonstration_prompt(train_dataset, 1)
# s += test_prompt_with_instruction(test_dataset[1])

# generate_and_print_text(model, s, tokenizer, max_new_tokens=100)

Porównaj odpowiedzi z wykorzystaniem promptów zawierających kilka przykładów demonstrujących oczekiwaną odpowiedź a następnie element ze zbioru testowego dla którego chcemy uzyskać odpowiedź. Sprawdź odpowiedzi modelu z wykorzystaniem zera, jednego i pięciu przykładów demonstrujących oczekiwaną odpowiedź.

In [28]:
# Zero przykładów demonstrujących oczekiwaną odpowiedź (zero-shot learning)
s = test_prompt_with_instruction(test_dataset[1])

generate_and_print_text(model, s, tokenizer, max_new_tokens=150, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
on april first i need a flight going from phoenix to san diego
### Meaning representation:
{
    "fromloc": "phoenix",
    "toloc": "san diego",
    "depart_date": {
        "day_name": "april",
        "day_number": 1,
        "day_name": "first",
        "month_name": "april",
        "relative": "first",
        "time": "first",
        "period_of_day": "first",
        "depart_date": {
            "day_name": "first",
            "day_number": 1,
            "day_name": "first",
            "month_name": "april",


In [29]:
# Jeden przykład demonstrujący oczekiwaną odpowiedź (1-shot learning)
s = create_demonstration_prompt(train_dataset, 1)
s += test_prompt_with_instruction(test_dataset[1])

generate_and_print_text(model, s, tokenizer, max_new_tokens=100, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
may i have a listing of flights from milwaukee to tacoma washington departing in the evening
### Meaning representation:
{
    "depart_time.period_of_day": "evening",
    "fromloc.city_name": "milwaukee",
    "toloc.city_name": "tacoma"
}
---
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
on april first i need a flight going from phoen

In [30]:
# Pięć przykładów demonstrujących oczekiwaną odpowiedź (5-shot learning)
s = create_demonstration_prompt(train_dataset, 5)
s += test_prompt_with_instruction(test_dataset[1])

generate_and_print_text(model, s, tokenizer, max_new_tokens=100, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
do you have flights from st. petersburg to toronto on monday
### Meaning representation:
{
    "depart_date.day_name": "monday",
    "fromloc.city_name": "st. petersburg",
    "toloc.city_name": "toronto"
}
---
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
what is the cheapest flight from boston to san francisco
### Meaning representa

#Zadanie 2 - Wybrana metoda efektywnego dostrojenia modelu

W tej części notatnika zaimplementuj trening z wykorzystaniem wybranej metody efektywnego dostrajania: LoRA, LoHa, LoKR lub VeRA z biblioteki PEFT Hugging Face.
Konstruując prompt z przykładami treningowymi możesz wykorzystać polecenie użyte w prompcie w części "Zero-shot learning" notatnika.

Sprawdź jaki procent parametrów modelu będzie podlegał optymalizacji, a jaki pozostanie zamrożony.

Wytrenuj model stosując zaimplementowaną metodę. Porównaj odpowiedzi z modelu bazowego i modelu dostrojonego na kilku przykładach ze zbioru testowego.

In [31]:
model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=q_config, torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True)
print(model)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2048)
    (layers): ModuleList(
      (0-23): 24 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (dense): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear8bitLt(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear8bitLt(in_features=8192, out_features=2048, bias=True)
        )
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_la

Utworzenie instancji modelu przygotowanej do dostrojenia metodą LoRA.

**Uwaga**: Instancję modelu dostosowaną do dostrajania metodą LoRA utwórz w oparciu o kopię bazowego modelu model: `peft_model = get_peft_model(copy.deepcopy(model), config)`. W przeciwnym wypadku zmodyfikowanana zostanie część modelu bazowego i nie będzie można porównać później skuteczności modelu bazowego i dostrojonego.

In [32]:
import copy


from peft import LoraConfig, get_peft_model, TaskType

config = LoraConfig(r=8, lora_alpha=16, target_modules=["Wqkv", "fc1", "fc2"],
                    bias="none", lora_dropout=0.05, task_type=TaskType.CAUSAL_LM)

# Utworzenie modelu do efektywnego dostrajania bazując na kopii modelu
peft_model = get_peft_model(copy.deepcopy(model), config)

Sprawdzenie liczby dostrajanych parametrów w modelu trenowanym z wykorzystaniem metody LoRA.

In [33]:
peft_model.print_trainable_parameters()

trainable params: 3,932,160 || all params: 1,422,202,880 || trainable%: 0.2765


Dostrojenie modelu `peft_model` z wykorzystaniem klasy Trainer.

In [34]:
def generate_and_tokenize_train_prompt_with_padding(data_point):
    max_length = 180
    return tokenize_with_padding(train_prompt_with_instruction(data_point), max_length)

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_train_prompt_with_padding)

splitted_tokenized_train_dataset = tokenized_train_dataset.train_test_split(test_size=0.15)
tokenized_train_dataset = splitted_tokenized_train_dataset["train"]
tokenized_val_dataset = splitted_tokenized_train_dataset["test"]

print(f"tokenized_train_dataset size: {tokenized_train_dataset.shape}")
print(f"tokenized_val_dataset size: {tokenized_val_dataset.shape}")

tokenized_train_dataset size: (3116, 8)
tokenized_val_dataset size: (550, 8)


In [35]:
import transformers
from datetime import datetime

output_dir = "./phi-qlora"

report_to = "none"
#report_to = "wandb"

trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=3,
        per_device_train_batch_size=6,
        gradient_accumulation_steps=2,
        max_steps=400,
        learning_rate=2.5e-5,
        optim="paged_adamw_8bit",
        logging_dir="./logs",        # Directory for storing logs
        logging_steps = 100,
        eval_strategy="steps",       # Evaluate the model every logging step
        eval_steps=50 ,              # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to=report_to
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

peft_model.config.use_cache = False
trainer.train()
peft_model.config.use_cache = True

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss,Validation Loss
50,No log,1.377261
100,1.318618,0.509546
150,1.318618,0.316354
200,0.333598,0.265015
250,0.333598,0.247218
300,0.241289,0.236182
350,0.241289,0.230970
400,0.226481,0.229113


Porównaj opowiedzi z bazowej, niedostrojonej wersji modelu (`model`) oraz dostrojonej wersji modelu (`peft_model`) dla kilku przykładowych elementów z testowego zbioru danych.

- Example 1

In [36]:
prompt1 = test_prompt_with_instruction(test_dataset[1])

In [37]:
generate_and_print_text(peft_model, prompt1, tokenizer, max_new_tokens=100, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
on april first i need a flight going from phoenix to san diego
### Meaning representation:
{
    "fromloc.city_name": "phoenix",
    "toloc.city_name": "san diego"
}
---
### Meaning representation:
{
    "depart_date.day_name": "first",
    "depart_time.period_of_day": "morning"
}
---
### Meaning representation:
{
    "depart_date.day_number": "first",


Model response after filtering: 

{
    "fromloc.city_name": "phoenix",
    "toloc.city_name": "san diego"
}


In [38]:
generate_and_print_text(model, prompt1, tokenizer, max_new_tokens=100, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
on april first i need a flight going from phoenix to san diego
### Meaning representation:
{
    "fromloc": "phoenix",
    "toloc": "san diego",
    "depart_date": {
        "day_name": "april",
        "day_number": 1,
        "day_name": "first",
        "month_name": "april",
        "relative": "first",
        "time": "first",
        "period_of_day": "first


- Example 2

In [39]:
prompt2 = test_prompt_with_instruction(test_dataset[123])

In [40]:
generate_and_print_text(peft_model, prompt2, tokenizer, max_new_tokens=100, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
what united airlines flights on june eighth go from westchester county to cincinnati
### Meaning representation:
{
    "fromloc.city_name": "westchester county",
    "toloc.city_name": "cincinnati"
}
---
### Meaning representation:
{
    "fromloc.city_name": "westchester county"
    "toloc.city_name": "cincinnati"
}
---
### Meaning representation:
{
    "fromloc.city_name": "westchester county"
    


Model response after filtering: 

{
    "fromloc.city_name": "westchester county",
    "toloc.city_name": "cincinnati"
}


In [41]:
generate_and_print_text(model, prompt2, tokenizer, max_new_tokens=100, filter_json=True)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.day_name', 'depart_date.month_name', 'depart_time.time', 'depart_date.relative', 'airline_name']. Not all keys are required.
### Input sentence:
what united airlines flights on june eighth go from westchester county to cincinnati
### Meaning representation:
{
    "fromloc": "westchester county",
    "toloc": "cincinnati",
    "depart_date": {
        "day_name": "june eighth",
        "day_number": 8,
        "day_name": "go",
        "month_name": "June",
        "relative": "on",
        "time": "8:00 PM",
        "period_of_day


#Zadanie 3 - Wybrana metoda dostrojenia promptu (tu: prompt tuning)

W tej części notatnika zaimplementuj trening z wykorzystaniem wybranej metody dostrajania promptu: prompt tuning, prefix tuning lub p-tuning z biblioteki PEFT Hugging Face.

Sprawdź jaki procent parametrów będzie podlegał optymalizacji, a jaki pozostanie zamrożony.

Wytrenuj model stosując zaimplementowaną metodę. Porównaj odpowiedzi z modelu bazowego i modelu dostrojonego na kilku przykładach ze zbioru testowego.

Wygeneruj ponownie zbiór treningowy i walidacyjny. Nie dodawaj polecenia "Convert the following text to a JSON format..." do promptów w zbiorze treningowym.
Zamiast dodawać polecenie w języku naturalnym wykorzystamy wirtualny prompt optymalizowany w procesie uczenia.

In [42]:
# Zwolnienie pamięci
import gc

del model
del peft_model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [43]:
def train_prompt(data_point) -> str:
  prompt =f"""### Input sentence:
{data_point["text"]}
### Meaning representation:
{data_point["json"]}
---"""
  return prompt


def test_prompt(data_point) -> str:
  prompt =f"""### Input sentence:
{data_point["text"]}
### Meaning representation:
"""
  return prompt

In [44]:
print(f"Prompt wykorzystywany w treningu modelu:")
print(train_prompt(train_dataset[1]))
print("")

print(f"Prompt wykorzystywany podczas inferencji:")
print(test_prompt(train_dataset[1]))
# prompt brany z train_dataset tylko żeby porównać konwencje promptów treningowych i testowych na tym samym przykadzie, niżej do inferencji użyty już przykład z test_dataset
print("")

Prompt wykorzystywany w treningu modelu:
### Input sentence:
what flights are available from pittsburgh to baltimore on thursday morning
### Meaning representation:
{
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning",
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore"
}
---

Prompt wykorzystywany podczas inferencji:
### Input sentence:
what flights are available from pittsburgh to baltimore on thursday morning
### Meaning representation:




Ponownie wygeneruj stokenizowany zbiór treningowy i testowy korzystając z nowego promptu.

In [45]:
## jeśli korzystając z nowego promptu to chodzi pewnie o zbiór walidacyjny (nie testowy)

def generate_and_tokenize_train_prompt_without_instruction_with_padding(data_point):
    max_length = 92
    return tokenize_with_padding(train_prompt(data_point), max_length)

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_train_prompt_without_instruction_with_padding)

splitted_tokenized_train_dataset = tokenized_train_dataset.train_test_split(test_size=0.15)
tokenized_train_dataset = splitted_tokenized_train_dataset["train"]
tokenized_val_dataset = splitted_tokenized_train_dataset["test"]

print(f"tokenized_train_dataset size: {tokenized_train_dataset.shape}")
print(f"tokenized_val_dataset size: {tokenized_val_dataset.shape}")

Map:   0%|          | 0/3666 [00:00<?, ? examples/s]

tokenized_train_dataset size: (3116, 8)
tokenized_val_dataset size: (550, 8)


Utworzenie instancji modelu przygotowanej do dostrojenia metodą *prompt tuning*. W początkowym prompcie podlegającym optymalizacji warto umieścić listę kluczy JSON w wynikowym tekście (zmienna `attributes_to_keep`). Przykładowy początkowy prompt mógłby wyglądać następująco:

```
initial_prompt = f"{attributes_to_keep} xxx placeholder for virtual prompt xxx"
```


**Uwaga**: Instancję modelu dostosowaną do dostrajania utwórz w
oparciu o kopię bazowego modelu model: `peft_model = get_peft_model(copy.deepcopy(model), config)`. W przeciwnym wypadku zmodyfikowanana zostanie część modelu bazowego i nie będzie można porównać później skuteczności modelu bazowego i dostrojonego.

In [46]:
from peft import PromptEmbedding, PromptTuningConfig, PromptTuningInit, TaskType


def tokenize(prompt: str) -> dict[str, Any]:
    result = tokenizer(prompt)
    # Tokenizator zwraca słownik z polami input_ids i attention_mask.
    # Modele językowe z rodziny CausalLM oczekują, że tekst po tokenizacji będzie podany w polu 'labels'
    result["labels"] = result["input_ids"].copy()
    return result


# Inicjalizacja wirtualnego promptu
initial_prompt = f"{attributes_to_keep} xxx placeholder for virtual prompt xxx"
t = tokenize(initial_prompt)
initial_prompt_len = len(t['input_ids'])
print(f"Initial prompt len: {initial_prompt_len}")

peft_config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, prompt_tuning_init=PromptTuningInit.TEXT, prompt_tuning_init_text=initial_prompt,
                                 tokenizer_name_or_path=base_model_id, num_virtual_tokens=initial_prompt_len)

#peft_config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, prompt_tuning_init=PromptTuningInit.RANDOM, num_virtual_tokens=initial_prompt_len)


Initial prompt len: 99


In [47]:
print(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True)
peft_model = get_peft_model(copy.deepcopy(model), peft_config)
print(peft_model)

microsoft/phi-1_5


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): PhiForCausalLM(
    (model): PhiModel(
      (embed_tokens): Embedding(51200, 2048)
      (layers): ModuleList(
        (0-23): 24 x PhiDecoderLayer(
          (self_attn): PhiAttention(
            (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (k_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (v_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (dense): Linear(in_features=2048, out_features=2048, bias=True)
          )
          (mlp): PhiMLP(
            (activation_fn): NewGELUActivation()
            (fc1): Linear(in_features=2048, out_features=8192, bias=True)
            (fc2): Linear(in_features=8192, out_features=2048, bias=True)
          )
          (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
      )
      (rotary_emb): PhiRotaryEmbedding()
      (embed_dropout):

In [48]:
print(peft_model.print_trainable_parameters())

trainable params: 202,752 || all params: 1,418,473,472 || trainable%: 0.0143
None


Dostrojenie modelu `peft_model` z wykorzystaniem klasy `Trainer`.

In [49]:
trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=3,
        per_device_train_batch_size=2,
        fp16=True,
        gradient_accumulation_steps=4,
        max_steps=800,
        learning_rate=1e-4,
        optim="adafactor",
        logging_dir="./logs",        # Directory for storing logs
        logging_steps = 200,
        eval_strategy="steps",       # Evaluate the model every logging step
        eval_steps=200,              # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to='none'
   ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

peft_model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()
peft_model.config.use_cache = True

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss,Validation Loss
200,1.149676,0.727092
400,0.662875,0.648694
600,0.623864,0.625046
800,0.604137,0.614458


Odpowiedź z podstawowej wersji modelu `model`.

In [50]:
prompt = test_prompt(test_dataset[231])
model.to(device)
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=60)

### Input sentence:
i would like flights from san francisco to long beach
### Meaning representation:

# 1. "I would like flights from San Francisco to Long Beach"
# 2. "I would like flights from San Francisco to Long Beach"
# 3. "I would like flights from San Francisco to Long Beach"
# 4. "I would like flights from San Francisco


Odpowiedź z wersji z dostrojonym promptem modelu `peft_model`.

In [51]:
prompt = test_prompt(test_dataset[231])
generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=60, filter_json=True)

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2219: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn(


### Input sentence:
i would like flights from san francisco to long beach
### Meaning representation:
{
    "fromloc.city_name": "san francisco",
    "toloc.city_name": "long beach"
}
---

```python
# Exercise 2:

# Define the meaning representation for the following sentence:
#


Model response after filtering: 

{
    "fromloc.city_name": "san francisco",
    "toloc.city_name": "long beach"
}
